# 02. MLP·랜덤 포레스트 학습 및 검증

입력: `data/raw/`의 전국 시군구 학습 자료와 전주시 인구 자료. 다른 노트북을 먼저 실행할 필요가 없습니다.

출력: 모델별 행정동 예측, 검증 MAE/RMSE/R2, 시군구·행정동 분포 차이.

검증 수치는 전국 시군구 수준이며 행정동 예측 정확도를 직접 측정한 값이 아닙니다.

각 노트북은 별도 커널에서 실행할 수 있습니다. 공유 변수는 사용하지 않고 파일로 입력을 읽습니다. ZIP에는 기존 실행 결과도 들어 있습니다.

In [1]:
from pathlib import Path
import sys
# Supports Jupyter launched from the repository root or notebooks/.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('저장소 폴더 또는 notebooks 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.paths import ROOT, data_path, result_path

# 필요 시 주석 해제
# %pip install -r requirements.txt
from IPython.display import Image, HTML, display


In [2]:
"""PDF 기반 재구현. 원본 코드 복원이 아니며 설정/선정 규칙은 새로 명시했다."""
from pathlib import Path
import ast, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from scipy.spatial.distance import cdist
import folium
SEED = 42
PREDICTION_SOURCE = 'saved'
COMMERCIAL_DATASET = 'historical_2022'  # official_2025 선택 가능
COUNTS_SOURCE = 'saved'  # saved: 기존 집계 / rebuilt: 좌표 결과의 재집계
N_CLUSTERS = 3
REMOVE_OUTLIERS = False
OUTLIER_QUANTILE = 0.95
TARGET_DONG = '송천1동'  # PDF의 상세 분석 대상. 새 군집 결과와 별도로 명시

### `read_csv`

In [3]:
def read_csv(name):
    for enc in ('utf-8-sig', 'cp949'):
        try:
            return pd.read_csv(data_path(name), encoding=enc, low_memory=False).dropna(how='all')
        except UnicodeDecodeError:
            continue
    raise ValueError(f'인코딩 확인 필요: {name}')

### `numeric`

In [4]:
def numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

### `save_csv`

In [5]:
def save_csv(df, name):
    df.to_csv(result_path(name), index=False, encoding='utf-8-sig')

### `clean_dong`

In [6]:
def clean_dong(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.startswith('['):
        try:
            vals = ast.literal_eval(s)
            s = str(vals[0]) if vals else ''
        except (ValueError, SyntaxError):
            return None
    s = s.split()[-1] if s.split() else ''
    if s.lower() in ('nan', 'none', ''):
        return None
    return s

### `grouped_counts`

In [7]:
def grouped_counts(name, count_name):
    d = read_csv(name)
    d['행정동'] = d['ADDRESS'].map(clean_dong)
    audit = d[d['행정동'].isna()]
    save_csv(audit, name.replace('.csv', '_unmatched.csv'))
    return d.dropna(subset=['행정동']).groupby('행정동').size().rename(count_name).reset_index()

### `train_models`

In [8]:
def train_models(population):
    train = read_csv('인구_쓰레기배출_데이터.csv')
    cols = ['인구수', '세대수', '연간 쓰레기 배출량']
    for col in cols:
        train[col] = numeric(train[col])
    valid = train[cols].notna().all(axis=1) & (train[cols] >= 0).all(axis=1)
    save_csv(train[~valid], 'training_excluded.csv')
    train = train[valid].copy()
    if train['지역'].nunique() < 2:
        raise ValueError('지역 단위 검증에 필요한 지역이 부족합니다.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr, te = next(splitter.split(train, groups=train['지역']))
    X = train[['인구수', '세대수']]
    y = train['연간 쓰레기 배출량']
    models = {
        'random_forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=1),
        'mlp': make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32),
                    solver='lbfgs', max_iter=1500, random_state=SEED))
    }
    rows, predictions = [], {}
    # MLP의 target도 표준화하여 최적화 수치 규모를 안정화
    for name, model in models.items():
        mu = float(y.iloc[tr].mean()) if name == 'mlp' else 0.0
        sd = float(y.iloc[tr].std()) if name == 'mlp' else 1.0
        model.fit(X.iloc[tr], (y.iloc[tr] - mu) / sd)
        pred = model.predict(X.iloc[te]) * sd + mu
        rows.append({'model': name, 'MAE': mean_absolute_error(y.iloc[te], pred),
                     'RMSE': np.sqrt(mean_squared_error(y.iloc[te], pred)), 'R2': r2_score(y.iloc[te], pred),
                     'train_rows': len(tr), 'test_rows': len(te),
                     'test_regions': ','.join(sorted(train.iloc[te]['지역'].unique()))})
        mu = float(y.mean()) if name == 'mlp' else 0.0
        sd = float(y.std()) if name == 'mlp' else 1.0
        model.fit(X, (y - mu) / sd)
        q = population[['행정동', '인구수', '세대수']].copy()
        q['연간 쓰레기 배출량 예측 값'] = np.maximum(0, model.predict(q[['인구수','세대수']]) * sd + mu)
        predictions[name] = q
        save_csv(q, f'{name}_predictions.csv')
    save_csv(pd.DataFrame(rows), 'model_validation.csv')
    # 시군구 -> 행정동 분포 차이 확인. 검증 성능은 행정동 예측 정확도가 아님.
    save_csv(pd.DataFrame([{'feature': c, 'training_min': X[c].min(), 'training_max': X[c].max(),
                           'jeonju_min': population[c].min(), 'jeonju_max': population[c].max()} for c in X]),
             'domain_shift.csv')
    return predictions

## 모델 학습 및 결과 확인

In [9]:
population = read_csv('전북특별자치도 전주시_인구현황.csv').rename(columns={'구분':'행정동'})
for column in ['인구수', '세대수']:
    population[column] = numeric(population[column])
population['행정동'] = population['행정동'].map(clean_dong)
predictions = train_models(population)
display(pd.read_csv(result_path('model_validation.csv')))
display(predictions['random_forest'].head())
display(predictions['mlp'].head())

,model,MAE,RMSE,R2,train_rows,test_rows,test_regions
0,random_forest,11369.238658,15779.062873,0.963028,157,72,"강원,경기,대구,충남"
1,mlp,16786.747966,34446.800816,0.823797,157,72,"강원,경기,대구,충남"


,행정동,인구수,세대수,연간 쓰레기 배출량 예측 값
0,중앙동,9646,4885,8180.650304
1,풍남동,3549,2242,8180.650304
2,노송동,10362,5456,8180.650304
3,완산동,4881,2594,8180.650304
4,동서학동,5823,3113,8180.650304


,행정동,인구수,세대수,연간 쓰레기 배출량 예측 값
0,중앙동,9646,4885,5356.276063
1,풍남동,3549,2242,3865.590785
2,노송동,10362,5456,5543.957057
3,완산동,4881,2594,4179.549454
4,동서학동,5823,3113,4415.869800
